# Bead volume via disk integration - **batch version (many images at once)**

Side-view silhouette -> r(z) per pixel row -> `V = sum(pi * r_i^2 * dz)`

Same physics and same maths as the single-image notebook, but now:

- you select **any number of images** in one go (multi-select dialog, a folder, or a typed list)
- every image is processed with the same settings and collected into **one summary table**
  (one row per image: height, base radius, volume by disk integration, volume of the equivalent
  spherical cap, contact angle, ...)
- the per-row `r_i` tables for every image are kept too, in both cropped and full-image coordinates
- everything is written to a single Excel workbook (`bead_volumes.xlsx`) with a `summary` sheet
  plus one sheet per image, and to CSVs
- a set of plots for a shrinkage experiment: QC montage, overlaid `r(z)` profiles, normalised
  shape overlay, volume bar chart, **volume shrinkage / linear strain vs time**, and a
  disk-vs-spherical-cap agreement check

Two small robustness changes vs the single-image notebook:

1. **thresholding is done on the cropped ROI, not on the full frame.** Otsu picks its threshold
   from the pixels it is given, so cropping first stops the microscope's black surround and the
   on-screen text from dragging the threshold around. Each image therefore gets a threshold
   suited to itself.
2. the mask is **cleaned** (small closing + hole filling) before the largest blob is taken, so a
   glare spot inside the bead does not punch a hole in the silhouette and shrink the radius.

Run every cell **in order, top to bottom**.

In [ ]:
import os, re, glob, math, sys
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
!{sys.executable} -m pip install openpyxl
%matplotlib inline

## Step 0 - pick your images

Three ways to hand the notebook a set of images; the first one that yields files wins:

- **A** - type the paths into `IMAGE_PATHS`
- **B** - point `IMAGE_FOLDER` at a folder and every image in it is used
- **C** - leave both empty and a multi-select dialog opens (Ctrl-click / Shift-click to select
  many files at once)

Files are sorted *naturally*, so `t2` comes before `t10`. That order is the order used in every
table and plot below, and for shrinkage it is assumed to be the time order of the experiment.

In [ ]:
IMAGE_PATHS = [                 # A: paste paths here, e.g. r"C:\data\bead_t0.jpg",
]
IMAGE_FOLDER = None             # B: e.g. r"C:\data\bead_run_1"  (all images in the folder)
USE_FILE_DIALOG = True          # C: fall back to a multi-select dialog

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def natural_key(path):
    """Sort helper so bead_t2 comes before bead_t10."""
    name = os.path.basename(path).lower()
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", name)]

paths = [p for p in IMAGE_PATHS if p]

if not paths and IMAGE_FOLDER:
    paths = [p for p in glob.glob(os.path.join(IMAGE_FOLDER, "*"))
             if p.lower().endswith(EXTS)]

if not paths and USE_FILE_DIALOG:
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    paths = list(filedialog.askopenfilenames(
        title="Select ALL bead side-view images (Ctrl-click / Shift-click for multiple)",
        filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.tif *.tiff"), ("All files", "*.*")],
    ))
    root.destroy()

IMAGE_PATHS = sorted(dict.fromkeys(paths), key=natural_key)

if not IMAGE_PATHS:
    raise RuntimeError("No images selected - set IMAGE_PATHS or IMAGE_FOLDER, or re-run and pick files.")

print(f"{len(IMAGE_PATHS)} image(s) selected, in processing order:")
for i, p in enumerate(IMAGE_PATHS):
    print(f"  [{i}] {os.path.basename(p)}")

If the dialog never appears, check the taskbar (it can open behind the browser). If it is blocked
entirely, use option A or B in the cell above - e.g. `IMAGE_FOLDER = r"C:\path\to\folder"`.

## Parameters

`DEFAULT_ROI` applies to every image. If one image needs a different crop (the bead drifted, the
stage moved), add an entry to `ROI_OVERRIDES` keyed by the **file name**; only that image uses it.

`INVERT`, `MANUAL_THRESH` and `SCALE_PX_PER_UM` work exactly as in the single-image notebook.
Because each image is thresholded inside its own ROI, one `INVERT` setting normally covers a whole
run shot under the same lighting.

In [ ]:
SCALE_PX_PER_UM = None       # pixels per micron from the REFLECTED-path calibration.
                             # Leave as None to work in raw pixels (volumes then come out in px^3).
INVERT = False               # True if bead is dark-on-light; False if light-on-dark.
MANUAL_THRESH = None         # 0-255 to override Otsu auto-threshold, or None

# Region of interest = (x0, x1, y0, y1) in the FULL image's pixel coordinates.
# Cropped BEFORE thresholding, so overlay text / microscope frame never become "foreground".
# Set it tight enough to exclude the text and the empty background, and to cut the image off at
# the silicon mat surface (the bottom edge y1 acts as the contact line / baseline).
DEFAULT_ROI = (800, 1600, 700, 1075)

# Per-image exceptions, keyed by file name:
#   ROI_OVERRIDES = {"bead_t10.jpg": (820, 1620, 690, 1070)}
ROI_OVERRIDES = {}

# Mask cleanup before the largest blob is picked.
CLOSE_PX   = 5               # morphological closing kernel in px (0 = off) - seals thin gaps
FILL_HOLES = True            # fill glare holes inside the silhouette

# Optional time axis. If your file names carry the time point, give a regex with ONE capture
# group of digits, e.g. r"_t(\d+)min"  ->  bead_t30min.jpg gives t = 30.
# Leave as None to just use the image order (0, 1, 2, ...).
TIME_REGEX = None
TIME_UNIT  = "min"

# Which image is the un-shrunk reference for the strain calculation (0 = the first one).
REFERENCE_INDEX = 0

OUT_PREFIX = "bead_volumes"  # output files: bead_volumes.xlsx / _summary.csv / _profiles.csv

## Helper functions

`analyze_bead()` is the single-image pipeline from the original notebook wrapped into one function
so it can be applied to every image: crop -> threshold -> clean -> largest blob -> apex -> r(z)
profile -> volume. It returns the profile table, the derived numbers and the images needed for QC
plots, and it never raises for a single bad image - the failure is recorded and the batch goes on.

In [ ]:
def threshold_crop(img, invert=False, manual_thresh=None):
    """Otsu (or a manual level) on an already-cropped grayscale image."""
    blur = cv2.GaussianBlur(img, (5, 5), 0)
    if manual_thresh is not None:
        _, mask = cv2.threshold(blur, manual_thresh, 255, cv2.THRESH_BINARY)
    else:
        _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if invert:
        mask = cv2.bitwise_not(mask)
    return mask


def clean_mask(mask, close_px=5, fill_holes=True):
    """Seal thin gaps and fill interior holes (glare) so the silhouette stays solid."""
    if close_px and close_px > 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(close_px), int(close_px)))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    if fill_holes:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filled = np.zeros_like(mask)
        cv2.drawContours(filled, contours, -1, 255, thickness=cv2.FILLED)
        mask = filled
    return mask


def largest_component(mask):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        raise RuntimeError("No foreground blob found - check threshold/invert flag")
    idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return np.uint8(labels == idx) * 255


def bead_profile(mask, apex_y=None):
    """Per-row half-width (radius) of the mask, top to bottom, in the mask's own coordinates."""
    ys, xs = np.where(mask > 0)
    y_min, y_max = ys.min(), ys.max()
    if apex_y is not None:
        y_min = max(y_min, int(round(apex_y)))
    widths, rows, lefts, rights = [], [], [], []
    for y in range(y_min, y_max + 1):
        row_xs = xs[ys == y]
        if row_xs.size == 0:
            continue
        xl, xr = int(row_xs.min()), int(row_xs.max())
        rows.append(y); lefts.append(xl); rights.append(xr)
        widths.append(xr - xl + 1)
    return (np.array(rows), np.array(widths, dtype=float),
            np.array(lefts), np.array(rights))


def roi_for(path, default_roi=None, overrides=None):
    return (overrides or {}).get(os.path.basename(path), default_roi)


def time_for(path, index, regex=None):
    """Time point from the file name if TIME_REGEX matches, else the image index."""
    if regex:
        m = re.search(regex, os.path.basename(path))
        if m:
            return float(m.group(1))
    return float(index)


def analyze_bead(path, roi=None, invert=False, manual_thresh=None, scale=None,
                 close_px=5, fill_holes=True, apex_override=None):
    """Full single-image pipeline. Returns a dict with the profile table, the numbers and QC images."""
    img_full = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img_full is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    H, W = img_full.shape

    x0, x1, y0, y1 = roi if roi else (0, W, 0, H)
    x0, x1 = max(0, int(x0)), min(W, int(x1))
    y0, y1 = max(0, int(y0)), min(H, int(y1))
    if x1 - x0 < 2 or y1 - y0 < 2:
        raise ValueError(f"ROI {roi} does not overlap the image ({W}x{H})")

    img  = img_full[y0:y1, x0:x1]
    mask = threshold_crop(img, invert=invert, manual_thresh=manual_thresh)
    mask = clean_mask(mask, close_px=close_px, fill_holes=fill_holes)
    mask = largest_component(mask)

    ys, xs = np.where(mask > 0)
    apex_y = int(ys.min()) if apex_override is None else int(apex_override[1])
    apex_x = float(xs[ys == apex_y].mean()) if apex_override is None else float(apex_override[0])

    rows, widths_px, lefts, rights = bead_profile(mask, apex_y=apex_y)
    if rows.size == 0:
        raise RuntimeError("Empty profile - mask has no rows below the apex")

    r_px = widths_px / 2.0
    z_px = rows - rows.min()

    prof = pd.DataFrame({
        "z_px":          z_px.astype(int),
        "y_crop":        rows.astype(int),
        "Y_full":        (y0 + rows).astype(int),
        "x_left_full":   x0 + lefts,
        "x_right_full":  x0 + rights,
        "x_center_full": x0 + (lefts + rights) / 2.0,
        "width_px":      widths_px,
        "r_i_px":        r_px,
    })

    # --- volume: V = sum(pi * r_i^2 * dz), dz = 1 px ---------------------------------
    h_px     = float(len(rows))                 # height = number of rows, dz = 1 px
    a_px     = float(r_px[int(np.argmax(widths_px))])   # base radius = widest half-width
    V_px3    = float(np.pi * np.sum(r_px ** 2))
    V_cap_px3 = float((np.pi * h_px / 6.0) * (3 * a_px ** 2 + h_px ** 2))
    theta_deg = float(2 * math.degrees(math.atan2(h_px, a_px))) if a_px > 0 else np.nan

    m = {
        "image":        os.path.basename(path),
        "path":         path,
        "roi":          (x0, x1, y0, y1),
        "apex_x_crop":  apex_x,
        "apex_y_crop":  apex_y,
        "apex_X_full":  x0 + apex_x,
        "apex_Y_full":  y0 + apex_y,
        "n_rows":       int(len(rows)),
        "area_px":      int(np.count_nonzero(mask)),
        "h_px":         h_px,
        "a_px":         a_px,
        "width_max_px": float(widths_px.max()),
        "V_disk_px3":   V_px3,
        "V_cap_px3":    V_cap_px3,
        "cap_diff_pct": 100.0 * (V_px3 - V_cap_px3) / V_cap_px3 if V_cap_px3 else np.nan,
        "contact_angle_deg": theta_deg,
        "aspect_h_over_a":   h_px / a_px if a_px else np.nan,
    }

    if scale:
        prof["z_um"]   = prof["z_px"]   / scale
        prof["r_i_um"] = prof["r_i_px"] / scale
        m["h_um"]      = h_px / scale
        m["a_um"]      = a_px / scale
        m["V_disk_um3"] = V_px3 / scale ** 3
        m["V_cap_um3"]  = V_cap_px3 / scale ** 3
        m["V_disk_mm3"] = m["V_disk_um3"] / 1e9

    return {"metrics": m, "profile": prof, "img": img, "mask": mask,
            "apex": (apex_x, apex_y)}

## Step 1 - run the batch

Every image goes through the same pipeline. If one image fails (bad ROI, nothing found at the
chosen threshold) it is reported and skipped, and the rest still run.

In [ ]:
results, failures = [], []

for i, p in enumerate(IMAGE_PATHS):
    try:
        res = analyze_bead(
            p,
            roi=roi_for(p, DEFAULT_ROI, ROI_OVERRIDES),
            invert=INVERT,
            manual_thresh=MANUAL_THRESH,
            scale=SCALE_PX_PER_UM,
            close_px=CLOSE_PX,
            fill_holes=FILL_HOLES,
        )
        res["metrics"]["index"] = i
        res["metrics"]["label"] = os.path.splitext(os.path.basename(p))[0]
        res["metrics"]["time"]  = time_for(p, i, TIME_REGEX)
        results.append(res)
        m = res["metrics"]
        print(f"[{i}] {m['image']:<32s} h={m['h_px']:6.1f} px  a={m['a_px']:6.1f} px  "
              f"V={m['V_disk_px3']:.4e} px^3")
    except Exception as e:
        failures.append((p, repr(e)))
        print(f"[{i}] {os.path.basename(p):<32s} FAILED: {e}")

print(f"\n{len(results)} of {len(IMAGE_PATHS)} image(s) analysed successfully.")
if failures:
    print("Failed images (fix the ROI / INVERT / threshold for these and re-run):")
    for p, e in failures:
        print("  -", os.path.basename(p), "->", e)
if not results:
    raise RuntimeError("Nothing analysed - check DEFAULT_ROI and INVERT.")

## Step 1b - QC montage: **look at this before trusting any number**

Each panel is one image's crop with the detected silhouette outlined in red and the auto-detected
apex marked with a green `+`. Check that:

- the red outline hugs the bead, with no text, frame edge or reflection caught in it
- the green `+` sits on the real tip, not on a glare spot
- the bottom of the outline sits at the mat surface (the ROI's bottom edge), not above or below it

If one panel is wrong, fix that image with a `ROI_OVERRIDES` entry (or flip `INVERT`) and re-run
from Step 1.

In [ ]:
def montage(results, what="overlay", ncols=3, panel=3.4):
    n = len(results)
    ncols = min(ncols, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel * ncols, panel * nrows * 0.95),
                             squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, res in zip(axes.ravel(), results):
        m = res["metrics"]
        if what == "mask":
            ax.imshow(res["mask"], cmap="gray")
        else:
            ax.imshow(res["img"], cmap="gray")
            ax.contour(res["mask"], levels=[127], colors="#e34948", linewidths=1.2)
            ax.plot(*res["apex"], "+", color="#1baf7a", markersize=14, markeredgewidth=2.5)
        ax.set_title(f"[{m['index']}] {m['label']}", fontsize=9)
    fig.suptitle("Detected silhouette (red) and apex (green +)" if what != "mask"
                 else "Binary masks used for the volume integration", fontsize=11)
    fig.tight_layout()
    plt.show()

montage(results, what="overlay")
montage(results, what="mask")

## Step 2 - the summary table (one row per image)

`V_disk` is the disk-integration volume, `V_cap` the volume of the spherical cap with the same
height and base radius, and `cap_diff_pct` how far the real bead departs from that ideal cap -
a large value means the bead is not cap-shaped (slumped, pinned, or a bad mask).

With `SCALE_PX_PER_UM = None` the volumes are in **px^3**; set the scale and the µm/mm^3 columns
appear automatically.

In [ ]:
summary = pd.DataFrame([r["metrics"] for r in results])

base_cols = ["index", "label", "image", "time", "n_rows", "h_px", "a_px", "width_max_px",
             "V_disk_px3", "V_cap_px3", "cap_diff_pct", "contact_angle_deg",
             "aspect_h_over_a", "area_px", "apex_X_full", "apex_Y_full", "roi", "path"]
um_cols = [c for c in ["h_um", "a_um", "V_disk_um3", "V_cap_um3", "V_disk_mm3"] if c in summary]
summary = summary[[c for c in base_cols[:8] if c in summary] + um_cols +
                  [c for c in base_cols[8:] if c in summary]]

# which volume column the plots and the strain table use
VOL_COL  = "V_disk_um3" if "V_disk_um3" in summary else "V_disk_px3"
CAP_COL  = "V_cap_um3"  if "V_cap_um3"  in summary else "V_cap_px3"
LEN_UNIT = "um" if SCALE_PX_PER_UM else "px"
VOL_UNIT = f"{LEN_UNIT}^3"
H_COL, A_COL = ("h_um", "a_um") if SCALE_PX_PER_UM else ("h_px", "a_px")

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
summary.drop(columns=["path"])

## Step 3 - per-row `r_i` tables for every image

One long table holding every row of every bead, in both cropped and full-image coordinates
(`Y_full`, `x_left_full`, `x_right_full` are what you hover over in IrfanView on the **original**
photo). The same data also goes into one sheet per image in the workbook.

In [ ]:
profiles = pd.concat(
    [r["profile"].assign(index=r["metrics"]["index"],
                         label=r["metrics"]["label"],
                         image=r["metrics"]["image"])
     for r in results],
    ignore_index=True,
)
cols = ["index", "label", "image"] + [c for c in profiles.columns if c not in ("index", "label", "image")]
profiles = profiles[cols]

print(f"{len(profiles)} rows across {profiles['image'].nunique()} image(s)")
profiles.head(10)

### Save everything

`bead_volumes.xlsx` gets a `summary` sheet plus one sheet per image; the same content is written
as CSVs for anything that does not like Excel.

In [ ]:
def sheet_name(label, used):
    """Excel sheet names: <=31 chars, no []:*?/\\, and unique."""
    s = re.sub(r"[\[\]:*?/\\]", "_", str(label))[:31] or "sheet"
    base, k = s, 1
    while s in used:
        suffix = f"_{k}"
        s = base[:31 - len(suffix)] + suffix
        k += 1
    used.add(s)
    return s

xlsx = f"{OUT_PREFIX}.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as xl:
    summary.drop(columns=["path"]).to_excel(xl, sheet_name="summary", index=False)
    used = {"summary"}
    for r in results:
        r["profile"].to_excel(xl, sheet_name=sheet_name(r["metrics"]["label"], used), index=False)

summary.to_csv(f"{OUT_PREFIX}_summary.csv", index=False)
profiles.to_csv(f"{OUT_PREFIX}_profiles.csv", index=False)

print("saved:")
print(" ", xlsx, f"(summary + {len(results)} per-image sheet(s))")
print(" ", f"{OUT_PREFIX}_summary.csv")
print(" ", f"{OUT_PREFIX}_profiles.csv")

## Step 4 - plots

Colours: with up to 8 beads each one gets its own fixed colour; beyond that the beads are shaded
light-to-dark in processing order, since a long run is really a time sequence rather than 20
unrelated categories.

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
          "#e87ba4", "#008300", "#4a3aa7", "#e34948"]   # fixed categorical order, never cycled
INK, INK_SOFT, GRIDC = "#0b0b0b", "#52514e", "#d8d7d2"

def colors_for(n):
    if n <= len(SERIES):
        return SERIES[:n]
    cmap = plt.get_cmap("viridis")
    return [cmap(v) for v in np.linspace(0.12, 0.92, n)]

def tidy(ax, title=None, xlabel=None, ylabel=None, grid_axis="y"):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRIDC)
    ax.grid(True, axis=grid_axis, color=GRIDC, linewidth=0.8, alpha=0.9)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_SOFT, labelsize=9)
    if title:  ax.set_title(title, color=INK, fontsize=12, pad=10)
    if xlabel: ax.set_xlabel(xlabel, color=INK_SOFT, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK_SOFT, fontsize=10)
    return ax

COLORS = colors_for(len(results))
LABELS = [r["metrics"]["label"] for r in results]
print("plot colours assigned to:", ", ".join(LABELS))

### 4a - radius profiles `r(z)`, all beads on one axis

The shape of the whole bead, not just its volume. Beads that shrank uniformly stay geometrically
similar (same curve, smaller); a bead that only lost height, or that slumped outwards, shows up
here as a change in shape that a single volume number hides.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for res, c in zip(results, COLORS):
    p, m = res["profile"], res["metrics"]
    z = p["z_um"] if SCALE_PX_PER_UM else p["z_px"]
    r = p["r_i_um"] if SCALE_PX_PER_UM else p["r_i_px"]
    ax.plot(r, z, "-", color=c, linewidth=2, label=m["label"])
tidy(ax, "Bead radius profile r(z), apex at z = 0",
     f"r_i ({LEN_UNIT})", f"z below apex ({LEN_UNIT})", grid_axis="both")
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT, title="image", title_fontsize=9)
plt.tight_layout(); plt.show()

### 4b - normalised shape, `r/a` vs `z/h`

The same profiles with the size divided out. Curves that lie on top of each other mean the beads
are the **same shape** at different sizes - which is what pure isotropic shrinkage looks like, and
what justifies reading a linear strain off the volume ratio. The dashed line is a perfect
spherical cap for reference.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for res, c in zip(results, COLORS):
    p, m = res["profile"], res["metrics"]
    ax.plot(p["r_i_px"] / m["a_px"], p["z_px"] / m["h_px"], "-", color=c, linewidth=2,
            label=m["label"])

# reference spherical cap with the mean aspect ratio of the set
k = float(np.mean([r["metrics"]["aspect_h_over_a"] for r in results]))   # h/a
t = np.linspace(0, 1, 200)
R = (1 + k ** 2) / (2 * k)                    # sphere radius in units of a
ax.plot(np.sqrt(np.clip(R ** 2 - (R - k * t) ** 2, 0, None)), t,
        "--", color=INK_SOFT, linewidth=1.5, label=f"ideal cap (h/a={k:.2f})")

tidy(ax, "Normalised bead shape (size divided out)", "r / a", "z / h", grid_axis="both")
ax.invert_yaxis()
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

### 4c - volume per image

Disk integration against the spherical-cap volume computed from the same `h` and `a`. The two
bars should be close; where they are not, that bead's silhouette is not a spherical cap and the
disk integration - which makes no shape assumption - is the number to trust.

In [ ]:
x = np.arange(len(summary))
w = 0.36                      # < half the 0.40 offset, so the paired bars keep a visible gap
fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(summary) + 2), 5))
b1 = ax.bar(x - 0.20, summary[VOL_COL], w, color=SERIES[0], label="disk integration")
b2 = ax.bar(x + 0.20, summary[CAP_COL], w, color=SERIES[1], label="spherical cap (same h, a)")
for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.3g", fontsize=8, color=INK_SOFT, padding=2)
tidy(ax, f"Bead volume per image ({VOL_UNIT})", None, f"V ({VOL_UNIT})")
ax.set_xticks(x, summary["label"], rotation=20, ha="right")
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

### 4d - shrinkage and strain

The point of the measurement. Taking image `REFERENCE_INDEX` as the un-shrunk state:

- **volumetric shrinkage** = `1 - V/V0`
- **linear strain** = `(V/V0)^(1/3) - 1`, the isotropic-shrinkage equivalent of that volume loss
  (negative = contraction). This is the number that pairs with a strain measured any other way,
  and it is only meaningful if 4b showed the shape staying similar.

Two panels rather than two y-axes on one plot, so neither curve's scale distorts the other.

In [ ]:
if len(summary) < 2:
    print("Only one image - shrinkage needs at least two. Skipping.")
else:
    ref = summary.loc[summary["index"] == REFERENCE_INDEX]
    if ref.empty:
        ref = summary.iloc[[0]]
    V0 = float(ref[VOL_COL].iloc[0])
    ratio = summary[VOL_COL] / V0
    shrink_pct = 100.0 * (1 - ratio)
    lin_pct = 100.0 * (ratio ** (1 / 3) - 1)
    t = summary["time"].values
    xlabel = f"time ({TIME_UNIT})" if TIME_REGEX else "image index"

    fig, axes = plt.subplots(2, 1, figsize=(7.5, 8), sharex=True)
    axes[0].plot(t, shrink_pct, "-o", color=SERIES[0], linewidth=2, markersize=8)
    tidy(axes[0], f"Volume shrinkage relative to '{ref['label'].iloc[0]}'", None,
         "1 - V/V0  (%)", grid_axis="both")
    axes[1].plot(t, lin_pct, "-o", color=SERIES[1], linewidth=2, markersize=8)
    tidy(axes[1], "Equivalent linear strain  (V/V0)^(1/3) - 1", xlabel,
         "linear strain (%)", grid_axis="both")
    for ax in axes:
        ax.axhline(0, color=GRIDC, linewidth=1)
        ax.margins(x=0.08, y=0.15)
    for xi, yi, lab in zip(t, lin_pct, summary["label"]):
        axes[1].annotate(f"{yi:+.2f}%", (xi, yi), textcoords="offset points",
                         xytext=(0, 9), ha="center", fontsize=8, color=INK_SOFT)
    plt.tight_layout(); plt.show()

### 4e - height and base radius per image

A shrinking bead should lose height and base radius together. If only `h` falls while `a` holds,
the bead is pinned to the mat and shrinking anisotropically - the isotropic linear strain from 4d
then understates the vertical strain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)
for ax, col, name in zip(axes, [H_COL, A_COL], ["height h", "base radius a"]):
    bars = ax.bar(summary["label"], summary[col], color=SERIES[0], width=0.6)
    ax.bar_label(bars, fmt="%.1f", fontsize=8, color=INK_SOFT, padding=2)
    tidy(ax, f"{name} ({LEN_UNIT})", None, f"{name.split()[-1]} ({LEN_UNIT})")
    ax.tick_params(axis="x", labelrotation=20)
plt.tight_layout(); plt.show()

### 4f - disk integration vs spherical cap (agreement check)

Every bead as one point against the 1:1 line. Points on the line are cap-shaped; a point well off
it flags an image worth re-checking in the Step 1b montage before it goes into the strain fit.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6))
lo = float(min(summary[VOL_COL].min(), summary[CAP_COL].min()))
hi = float(max(summary[VOL_COL].max(), summary[CAP_COL].max()))
pad = 0.06 * (hi - lo or hi or 1)
ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "--", color=INK_SOFT, linewidth=1.5,
        label="1:1")
ax.scatter(summary[CAP_COL], summary[VOL_COL], s=90, color=SERIES[0],
           edgecolor="white", linewidth=1.5, zorder=3)
for _, row in summary.iterrows():
    ax.annotate(row["label"], (row[CAP_COL], row[VOL_COL]), textcoords="offset points",
                xytext=(8, 5), fontsize=8, color=INK_SOFT)
ax.margins(0.12)
tidy(ax, "Disk integration vs spherical-cap volume",
     f"V spherical cap ({VOL_UNIT})", f"V disk integration ({VOL_UNIT})", grid_axis="both")
ax.legend(frameon=False, fontsize=9, labelcolor=INK_SOFT)
plt.tight_layout(); plt.show()

## Step 5 - shrinkage / strain table

The same numbers as plot 4d in table form, appended to the workbook as a `shrinkage` sheet:

| column | meaning |
|---|---|
| `V_over_V0` | volume as a fraction of the reference bead |
| `shrinkage_pct` | `100 * (1 - V/V0)` - volume lost |
| `linear_strain_pct` | `100 * ((V/V0)^(1/3) - 1)` - isotropic linear strain, negative = contraction |
| `dV_pct_prev` | volume change relative to the **previous** image, for spotting a bad frame |

In [ ]:
ref_row = summary.loc[summary["index"] == REFERENCE_INDEX]
if ref_row.empty:
    ref_row = summary.iloc[[0]]
V0 = float(ref_row[VOL_COL].iloc[0])

shrink = summary[["index", "label", "time", H_COL, A_COL, VOL_COL]].copy()
shrink["V_over_V0"]         = shrink[VOL_COL] / V0
shrink["shrinkage_pct"]     = 100.0 * (1 - shrink["V_over_V0"])
shrink["linear_strain_pct"] = 100.0 * (shrink["V_over_V0"] ** (1 / 3) - 1)
shrink["dV_pct_prev"]       = 100.0 * shrink[VOL_COL].pct_change()

with pd.ExcelWriter(xlsx, engine="openpyxl", mode="a", if_sheet_exists="replace") as xl:
    shrink.to_excel(xl, sheet_name="shrinkage", index=False)
shrink.to_csv(f"{OUT_PREFIX}_shrinkage.csv", index=False)

print(f"reference = '{ref_row['label'].iloc[0]}',  V0 = {V0:.6g} {VOL_UNIT}")
print(f"saved shrinkage sheet to {xlsx} and {OUT_PREFIX}_shrinkage.csv")
shrink

## Notes / troubleshooting

- **A mask looks like a solid rectangle** -> flip `INVERT` and re-run from Step 1.
- **Text or frame caught in one mask** -> add that file to `ROI_OVERRIDES` with a tighter crop.
- **The bead's reflection in the mat is included** -> raise the ROI's bottom edge `y1` to the
  contact line; the integration stops at whatever `y1` you give it.
- **Volumes only mean something in µm^3 once `SCALE_PX_PER_UM` is set.** In pixels the *ratios*
  (and therefore the shrinkage and the strain) are still valid, since the scale cancels - so a
  strain analysis can be done without calibrating, an absolute volume cannot.
- **All images must share one scale and one working distance.** If the zoom changed between shots,
  the volumes are not comparable, and the shrinkage numbers are meaningless.
- `dz` is one pixel row, so the disk sum uses `dz = 1 px` and the volume comes out in px^3;
  dividing by `SCALE_PX_PER_UM^3` converts it, which is what the `_um3` columns do.